# Model 5: XGBoost (on v3 Features)

This notebook will test a new model, `XGBRegressor`, on our best feature set (`v3`). Our goal is to see if XGBoost can beat our best LightGBM local score of **36,197.72**.

XGBoost, like LightGBM, can perform quantile regression. It uses the objective `reg:quantileerror`.

## Setup
We will:
1.  Load the `v3` training and validation datasets.
2.  Define our custom evaluation metrics (`quantile_error_0_2_raw`).
3.  Define the special `eval_metric` wrapper that XGBoost requires.
4.  Split the data into `X_train`, `y_train`, `X_val`, and `y_val`.

In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Define the Evaluation Metric ---
def quantile_error_0_2_raw(y_true, y_pred):
    """
    Calculates the 0.2 Quantile Error
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred[y_pred < 0] = 0 # Ensure non-negative predictions
    loss = np.mean(np.maximum(0.2 * (y_true - y_pred), 0.8 * (y_pred - y_true)))
    return loss

def xgb_quantile_error_0_2(y_pred, y_true):
    """
    XGBoost-compatible wrapper for our metric.
    Note: XGBoost passes (predictions, labels)
    """
    # y_true is already a numpy array, not a DMatrix
    score = quantile_error_0_2_raw(y_true, y_pred)
    
    # --- THIS IS THE FIX ---
    # The function must return the metric name and the score as separate items.
    # The fix is to return the NAME as a string, and the VALUE as a float.
    # The previous error was returning a tuple AS the value.
    return 'q0.2_error', score

print("--- 1. Custom metrics defined ---")

# --- 2. Load v3 Datasets ---
try:
    df_train = pd.read_parquet("training_dataset_v3.parquet")
    df_val = pd.read_parquet("validation_dataset_v3.parquet")
    print(f"--- 2. Loaded 'training_dataset_v3.parquet' (Shape: {df_train.shape}) ---")
    print(f"---    Loaded 'validation_dataset_v3.parquet' (Shape: {df_val.shape}) ---")
except Exception as e:
    print(f"Error loading v3 files: {e}")
    raise

# --- 3. Split into Features (X) and Target (y) ---
FEATURE_COLS = [col for col in df_val.columns if col.startswith('f_')]
TARGET_COL = 'y_cumulative_weight'

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_val = df_val[FEATURE_COLS]
y_val = df_val[TARGET_COL]

print("--- 3. Split data into X_train, y_train, X_val, and y_val ---")
print(f"Total features being used: {len(FEATURE_COLS)}")

print("\n--- Setup Complete ---")

--- 1. Custom metrics defined ---
--- 2. Loaded 'training_dataset_v3.parquet' (Shape: (22644, 14)) ---
---    Loaded 'validation_dataset_v3.parquet' (Shape: (7050, 14)) ---
--- 3. Split data into X_train, y_train, X_val, and y_val ---
Total features being used: 11

--- Setup Complete ---


## Train Model 1 (Untuned XGBoost)

First, we will train an untuned XGBoost model. This will give us a direct comparison to our untuned LGBM v3 score of **40,945.52**.

We will set `objective='reg:quantileerror'` and `quantile_alpha=0.2`.

In [16]:
print("--- Model 1: XGBoost (Untuned) on v3 Features ---")

# --- 1. Baseline Score ---
y_pred_zero = np.zeros(len(y_val))
baseline_score = quantile_error_0_2_raw(y_val, y_pred_zero)
print(f"Baseline Score (predicting all zeros): {baseline_score:.2f}")

# --- 2. Switch to Native XGBoost API ---
print("\n--- 2. Training XGBoost model (Native API)... ---")
# We must use XGBoost's native DMatrix data structure
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Define parameters
params = {
    'objective': 'reg:quantileerror',
    'quantile_alpha': 0.2,
    'learning_rate': 0.05,
    'n_jobs': -1,
    'seed': 42,
    'eval_metric': 'rmse' # A default metric is still needed for logging
}

# This is the "wrapper" for the native xgb.train()
# It takes (y_pred, y_true_dmatrix)
def xgb_native_quantile_error(y_pred, y_true_dmatrix):
    # In the native API, the second argument IS a DMatrix
    y_true_labels = y_true_dmatrix.get_label()
    score = quantile_error_0_2_raw(y_true_labels, y_pred)
    # The native API wants a tuple: (metric_name, score_value)
    return 'q0.2_error', score

# Train the model using xgb.train
model_xgb_native = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'validation')],
    # --- THIS IS THE FIX ---
    # The 'feval' argument is deprecated. The correct argument is 'custom_metric'.
    custom_metric=xgb_native_quantile_error, 
    callbacks=[xgb.callback.EarlyStopping(rounds=50, 
                                          metric_name='q0.2_error', # This name must match our function's output
                                          maximize=False, 
                                          save_best=True)],
    verbose_eval=False # Suppress training logs
)

print("Training complete.")

# --- 3. Evaluate the Model ---
print("\n--- 3. Evaluating Model ---")
# Predict on our validation DMatrix
y_pred_xgb_v3_untuned = model_xgb_native.predict(dval)
model_score = quantile_error_0_2_raw(y_val, y_pred_xgb_v3_untuned)

# Get the untuned LGBM v3 score for comparison
lgbm_v3_untuned_score = 40945.52 # From 04_Model_LGBM_v3.ipynb

print(f"\n--- Results ---")
print(f"Baseline Score (predicting all zeros): {baseline_score:.2f}")
print(f"XGBoost v3 (Untuned) Score:              {model_score:.2f}")
print(f"LGBM v3 (Untuned) Score:                 {lgbm_v3_untuned_score:.2f}")

if model_score < lgbm_v3_untuned_score:
    print("\nUntuned XGBoost is BETTER than untuned LGBM.")
else:
    print("\nUntuned LGBM is better than untuned XGBoost.")

--- Model 1: XGBoost (Untuned) on v3 Features ---
Baseline Score (predicting all zeros): 54088.21

--- 2. Training XGBoost model (Native API)... ---
Training complete.

--- 3. Evaluating Model ---

--- Results ---
Baseline Score (predicting all zeros): 54088.21
XGBoost v3 (Untuned) Score:              37795.02
LGBM v3 (Untuned) Score:                 40945.52

Untuned XGBoost is BETTER than untuned LGBM.


## Model 2: Hyperparameter Tuning (XGBoost)

Our untuned XGBoost model gave us a score of **37,795.02**, which is over 3,000 points better than the untuned LGBM. This is a fantastic result and proves XGBoost is a better choice for our features.

Now, we will run `Optuna` on this new model. Our goal is to find a set of hyperparameters that can beat our all-time best score from the tuned LGBM (`36,197.72`).

In [17]:
import optuna

print(f"Optuna version: {optuna.__version__}")

# --- 1. Define the Objective Function (for XGBoost) ---
# We already have dtrain and dval from the previous cell
def objective_xgb_v3(trial):
    # Define the search space for our hyperparameters
    params = {
        'objective': 'reg:quantileerror',
        'quantile_alpha': 0.2,
        'n_jobs': -1,
        'seed': 42,
        'eval_metric': 'rmse', # Default metric
        
        # --- Parameters to TUNE ---
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10) # A key XGB param
    }
    
    # Train the model with these trial parameters
    model_opt = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, 'validation')],
        custom_metric=xgb_native_quantile_error, # Our wrapper from Cell 2
        callbacks=[xgb.callback.EarlyStopping(rounds=50, 
                                              metric_name='q0.2_error', 
                                              maximize=False, 
                                              save_best=True)],
        verbose_eval=False # Suppress training logs
    )
    
    # Get predictions and calculate the score
    y_pred_opt = model_opt.predict(dval)
    score = quantile_error_0_2_raw(y_val, y_pred_opt)
    
    return score

# --- 2. Run the Tuning Study ---
print("\n--- 2. Running Optuna study on XGBoost v3... ---")
study_xgb_v3 = optuna.create_study(direction='minimize')

# We'll run 50 trials again
study_xgb_v3.optimize(objective_xgb_v3, n_trials=50)

# --- 3. Report Best Results ---
print("\n--- Optuna XGB v3 Study Complete ---")
print(f"Number of finished trials: {len(study_xgb_v3.trials)}")
print("Best trial:")
best_trial_xgb_v3 = study_xgb_v3.best_trial

print(f"  Value (Best XGB Score): {best_trial_xgb_v3.value:.2f}")
print("  Params: ")
for key, value in best_trial_xgb_v3.params.items():
    print(f"    {key}: {value}")

# Let's compare to our previous best tuned score
LGBM_BEST_SCORE = 36197.72
print("\n--- Comparison ---")
print(f"Previous Best (Tuned LGBM v3): {LGBM_BEST_SCORE:.2f}")
print(f"New Best (Tuned XGB v3):       {best_trial_xgb_v3.value:.2f}")
print(f"Improvement:                   {LGBM_BEST_SCORE - best_trial_xgb_v3.value:.2f}")

/Users/jennarx/Desktop/NTNU/ModernMachineLearning/group_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-07 01:09:20,412] A new study created in memory with name: no-name-f9c00d28-776e-4208-ad4e-fe7695d0ab35


Optuna version: 4.5.0

--- 2. Running Optuna study on XGBoost v3... ---


[I 2025-11-07 01:09:20,675] Trial 0 finished with value: 38905.83359494036 and parameters: {'learning_rate': 0.07478498790041756, 'max_depth': 9, 'subsample': 0.7678659294403318, 'colsample_bytree': 0.708454630247431, 'min_child_weight': 10}. Best is trial 0 with value: 38905.83359494036.
[I 2025-11-07 01:09:20,902] Trial 1 finished with value: 38138.1782206143 and parameters: {'learning_rate': 0.04673063135520281, 'max_depth': 4, 'subsample': 0.8031998150367008, 'colsample_bytree': 0.8931737416642705, 'min_child_weight': 9}. Best is trial 1 with value: 38138.1782206143.
[I 2025-11-07 01:09:21,121] Trial 2 finished with value: 38431.65623052363 and parameters: {'learning_rate': 0.06489557188234416, 'max_depth': 6, 'subsample': 0.9173812327132324, 'colsample_bytree': 0.8885076797706174, 'min_child_weight': 4}. Best is trial 1 with value: 38138.1782206143.
[I 2025-11-07 01:09:21,425] Trial 3 finished with value: 39155.92925895748 and parameters: {'learning_rate': 0.02971218432761733, 'ma


--- Optuna XGB v3 Study Complete ---
Number of finished trials: 50
Best trial:
  Value (Best XGB Score): 35288.80
  Params: 
    learning_rate: 0.0757713467333453
    max_depth: 5
    subsample: 0.7759846554256271
    colsample_bytree: 0.7124590827781982
    min_child_weight: 4

--- Comparison ---
Previous Best (Tuned LGBM v3): 36197.72
New Best (Tuned XGB v3):       35288.80
Improvement:                   908.92


## Model 2b: Extended Hyperparameter Tuning (XGBoost)

Our 50-trial run was a huge success, giving us a new best score of **35,288.80**. As you noted, this was extremely fast.

We will now run a much more aggressive 200-trial search to see if we can find an even better combination of hyperparameters. The goal is to squeeze every last point of performance out of this model.

In [19]:
import optuna

print(f"Optuna version: {optuna.__version__}")

# --- 1. Define the Objective Function (for XGBoost) ---
# We already have dtrain, dval, and the metric function from previous cells
def objective_xgb_v3(trial):
    # We can use the same search space as before
    params = {
        'objective': 'reg:quantileerror',
        'quantile_alpha': 0.2,
        'n_jobs': -1,
        'seed': 42,
        'eval_metric': 'rmse', 
        
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    # Train the model with these trial parameters
    model_opt = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, 'validation')],
        custom_metric=xgb_native_quantile_error, 
        callbacks=[xgb.callback.EarlyStopping(rounds=50, 
                                              metric_name='q0.2_error', 
                                              maximize=False, 
                                              save_best=True)],
        verbose_eval=False 
    )
    
    # Get predictions and calculate the score
    y_pred_opt = model_opt.predict(dval)
    score = quantile_error_0_2_raw(y_val, y_pred_opt)
    
    return score

# --- 2. Run the EXTENDED Tuning Study ---
print("\n--- 2. Running EXTENDED (200 trial) Optuna study on XGBoost v3... ---")
# This will take approx. 4x longer (1-2 minutes), which is fine.
study_xgb_v3_extended = optuna.create_study(direction='minimize')

study_xgb_v3_extended.optimize(objective_xgb_v3, n_trials=200) # <-- Increased from 50 to 200

# --- 3. Report Best Results ---
print("\n--- Optuna XGB v3 (200-Trial) Study Complete ---")
print(f"Number of finished trials: {len(study_xgb_v3_extended.trials)}")
print("Best trial:")
best_trial_xgb_v3_ext = study_xgb_v3_extended.best_trial

print(f"  Value (Best XGB Score): {best_trial_xgb_v3_ext.value:.2f}")
print("  Params: ")
for key, value in best_trial_xgb_v3_ext.params.items():
    print(f"    {key}: {value}")

# Let's compare to our 50-trial run
PREV_BEST_SCORE = 35288.80
print("\n--- Comparison ---")
print(f"Previous Best (Tuned XGB, 50 trials): {PREV_BEST_SCORE:.2f}")
print(f"New Best (Tuned XGB, 200 trials):     {best_trial_xgb_v3_ext.value:.2f}")
print(f"Improvement:                          {PREV_BEST_SCORE - best_trial_xgb_v3_ext.value:.2f}")

[I 2025-11-07 01:11:43,446] A new study created in memory with name: no-name-a8b850f5-bf49-479a-a33c-9d9008f39a79


Optuna version: 4.5.0

--- 2. Running EXTENDED (200 trial) Optuna study on XGBoost v3... ---


[I 2025-11-07 01:11:43,741] Trial 0 finished with value: 37217.59453071553 and parameters: {'learning_rate': 0.049173195942487556, 'max_depth': 6, 'subsample': 0.8540685147436871, 'colsample_bytree': 0.8955348121004703, 'min_child_weight': 8}. Best is trial 0 with value: 37217.59453071553.
[I 2025-11-07 01:11:44,335] Trial 1 finished with value: 37420.41620335765 and parameters: {'learning_rate': 0.012672970739794178, 'max_depth': 6, 'subsample': 0.7858250703527445, 'colsample_bytree': 0.7742835644301054, 'min_child_weight': 2}. Best is trial 0 with value: 37217.59453071553.
[I 2025-11-07 01:11:44,724] Trial 2 finished with value: 38435.25657976846 and parameters: {'learning_rate': 0.026244552565933114, 'max_depth': 7, 'subsample': 0.9565298310728854, 'colsample_bytree': 0.9134340172685348, 'min_child_weight': 2}. Best is trial 0 with value: 37217.59453071553.
[I 2025-11-07 01:11:45,001] Trial 3 finished with value: 36852.82238012017 and parameters: {'learning_rate': 0.0394200237003684


--- Optuna XGB v3 (200-Trial) Study Complete ---
Number of finished trials: 200
Best trial:
  Value (Best XGB Score): 33980.51
  Params: 
    learning_rate: 0.099060649608946
    max_depth: 6
    subsample: 0.8172419290767476
    colsample_bytree: 0.7786050937155478
    min_child_weight: 7

--- Comparison ---
Previous Best (Tuned XGB, 50 trials): 35288.80
New Best (Tuned XGB, 200 trials):     33980.51
Improvement:                          1308.29


## Phase 7: Creating Final XGBoost Submission (v3)

Our aggressive 200-trial Optuna study was a complete success, finding an
excellent new local score of **33,980.51**.

We will now create our final submission file based on this best-tuned XGBoost model.

This script will:
1.  Get the best-tuned parameters from our `study_xgb_v3_extended` object.
2.  Find the optimal `n_estimators` (best iteration) for this new model.
3.  Re-train the model on 100% of our `v3` data (training + validation).
4.  Build the `v3` test features from scratch using all raw files.
5.  Generate predictions and save them to `xgboost.csv` with the correct `predicted_weight` column.

In [24]:
import pandas as pd
import numpy as np
import xgboost as xgb
import itertools

print("--- Phase 7: Generating Tuned Kaggle Submission (XGBoost v3) ---")

# --- 1. Get Best Tuned Parameters ---
try:
    best_params = study_xgb_v3_extended.best_trial.params.copy()
    print(f"Loaded best parameters from 200-trial Optuna study: {best_params}")
    
    # Add the fixed parameters
    best_params['objective'] = 'reg:quantileerror'
    best_params['quantile_alpha'] = 0.2
    best_params['n_jobs'] = -1
    best_params['seed'] = 42
    best_params['eval_metric'] = 'rmse' # Default metric

    print("\nFinding best n_estimators for tuned model...")
    # We can re-use the dtrain/dval and metric function from previous cells
    model_tuned_temp = xgb.train(
        best_params,
        dtrain,
        num_boost_round=2000, # Give it plenty of room
        evals=[(dval, 'validation')],
        custom_metric=xgb_native_quantile_error,
        callbacks=[xgb.callback.EarlyStopping(rounds=50, 
                                              metric_name='q0.2_error', 
                                              maximize=False, 
                                              save_best=True)],
        verbose_eval=False
    )
    
    BEST_TUNED_ITERATION = model_tuned_temp.best_iteration
    print(f"Found best iteration for tuned model: {BEST_TUNED_ITERATION}")
    
except NameError:
    print("ERROR: 'study_xgb_v3_extended' or 'dtrain'/'dval' objects not found.")
    print("Please re-run the Optuna cell (Cell 8) first.")
    raise

# --- 2. Combine & Re-train Best Tuned Model ---
print("\n--- 2. Re-training tuned model on all data ---")

# Load all v3 data
df_train_full = pd.read_parquet("training_dataset_v3.parquet")
df_val_full = pd.read_parquet("validation_dataset_v3.parquet")
df_full = pd.concat([df_train_full, df_val_full], ignore_index=True)

FEATURE_COLS_FULL = [col for col in df_full.columns if col.startswith('f_')]
TARGET_COL_FULL = 'y_cumulative_weight'

X_full = df_full[FEATURE_COLS_FULL]
y_full = df_full[TARGET_COL_FULL]

# Create the full DMatrix for final training
dfull = xgb.DMatrix(X_full, label=y_full)

# Update params with the final number of estimators
best_params['n_estimators'] = BEST_TUNED_ITERATION

print(f"Training on full dataset with {BEST_TUNED_ITERATION} rounds...")
model_final_tuned = xgb.train(
    best_params,
    dfull,
    num_boost_round=BEST_TUNED_ITERATION # Train for the exact best rounds
)
print("Final tuned model trained.")


# --- 3. Load Raw Data for Test Set Generation ---
print("\n--- 3. Loading all raw data for test set generation ---")
try:
    df_receivals = pd.read_csv("data/kernel/receivals.csv")
    df_po = pd.read_csv("data/kernel/purchase_orders.csv")
    df_materials = pd.read_csv("data/extended/materials.csv")
    df_transport = pd.read_csv("data/extended/transportation.csv")
    df_mapping = pd.read_csv("data/prediction_mapping.csv")
    print("All raw files loaded.")
except Exception as e:
    print(f"Error loading raw files: {e}")
    raise

# --- 4. Re-run v3 Feature Engineering for Test Set ---
print("--- 4. Cleaning raw data ---")
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
df_receivals_cleaned = df_receivals_cleaned[df_receivals_cleaned['net_weight'] > 0].copy()

material_map = df_materials[['product_id', 'rm_id', 'raw_material_format_type']].drop_duplicates()
product_to_rm_map = material_map.drop_duplicates(subset=['product_id'], keep='first')

df_transport_cleaned = df_transport[
    ['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name', 'net_weight']
].dropna(subset=['transporter_name']).copy()
df_transport_cleaned.rename(columns={'net_weight': 'transport_net_weight'}, inplace=True)
df_transport_cleaned = df_transport_cleaned.drop_duplicates(
    subset=['rm_id', 'purchase_order_id', 'purchase_order_item_no'], 
    keep='last'
)

df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce', utc=True)
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce', utc=True)
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit', 'created_date_time']).copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['quantity'] > 0].copy()
df_po_cleaned = pd.merge(df_po_cleaned, product_to_rm_map, on='product_id', how='left')
df_po_cleaned = df_po_cleaned.dropna(subset=['rm_id'])

df_mapping['forecast_end_date'] = pd.to_datetime(df_mapping['forecast_end_date'], utc=True)
df_test = df_mapping.copy()

print(f"Test set grid created. Shape: {df_test.shape}")

print("--- 5. Building test set features (v3) ---")
TEST_START_DATE = pd.to_datetime('2025-01-01', utc=True)
hist_receivals = df_receivals_cleaned.copy() 
hist_po = pd.merge(
    df_po_cleaned,
    df_transport_cleaned[['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name']],
    on=['rm_id', 'purchase_order_id', 'purchase_order_item_no'],
    how='left'
)
hist_po['transporter_name'] = hist_po['transporter_name'].fillna('Transporter_Unknown')

# Aggregated PO Features
print("Building feature (f_cumulative_po_quantity)...")
hist_po['f_po_lead_time_days'] = (hist_po['delivery_date'] - hist_po['created_date_time']).dt.days
po_agg = hist_po.groupby(['rm_id', 'delivery_date']).agg(
    daily_po_quantity=('quantity', 'sum'),
    avg_lead_time=('f_po_lead_time_days', 'mean')
).reset_index()
po_agg.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
rm_ids_to_merge = df_test['rm_id'].unique()
merged_df = pd.merge(
    df_test[['rm_id', 'forecast_end_date']],
    po_agg[po_agg['rm_id'].isin(rm_ids_to_merge)],
    on='rm_id', how='left'
)
merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
cumulative_features = merged_df_filtered.groupby(['rm_id', 'forecast_end_date']).agg(
    f_cumulative_po_quantity=('daily_po_quantity', 'sum'),
    f_avg_lead_time=('avg_lead_time', 'mean')
).reset_index()
df_test = pd.merge(df_test, cumulative_features, on=['rm_id', 'forecast_end_date'], how='left')
df_test['f_cumulative_po_quantity'] = df_test['f_cumulative_po_quantity'].fillna(0)
df_test['f_avg_lead_time'] = df_test['f_avg_lead_time'].fillna(0)

# Time-Based
print("Building features (Time-Based)...")
df_test['f_month'] = df_test['forecast_end_date'].dt.month
df_test['f_day_of_week'] = df_test['forecast_end_date'].dt.dayofweek
df_test['f_day_of_year'] = df_test['forecast_end_date'].dt.dayofyear
df_test['f_is_month_end'] = df_test['forecast_end_date'].dt.is_month_end.astype(str)

# Entity & Lag
print("Building features (Entity & Lag)...")
df_train_merged_eda = pd.merge(
    hist_receivals,
    hist_po[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
    on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'], how='inner'
)
df_train_merged_eda['delivery_lag_days'] = (
    df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
).dt.days
rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
df_test = pd.merge(df_test, rm_id_lag_map, on='rm_id', how='left')
df_test['f_median_lag_days'] = df_test['f_median_lag_days'].fillna(0)

# --- THIS IS THE FIX ---
# f_receivals_Nd
# We will write this loop more robustly to avoid any f-string parsing errors.
print("Building features (f_receivals_Nd)...")
hist_windows = [30, 90, 180]
for days in hist_windows:
    hist_start_date_window = TEST_START_DATE - pd.Timedelta(days=days)
    window_data = hist_receivals[
        (hist_receivals['date_arrival'] >= hist_start_date_window) &
        (hist_receivals['date_arrival'] < TEST_START_DATE)
    ]
    
    # Create the feature name
    feature_name = f'f_receivals_{days}d'
    
    feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=feature_name)
    df_test = pd.merge(df_test, feature_map, on='rm_id', how='left')
    
    # Fill NaNs using the explicit feature name
    df_test[feature_name] = df_test[feature_name].fillna(0)

# V3 Static Features
rm_id_static_features = hist_po.drop_duplicates(subset=['rm_id'], keep='last')[[
    'rm_id', 'status', 'raw_material_format_type', 'transporter_name'
]]
df_test = pd.merge(df_test, rm_id_static_features, on='rm_id', how='left')
df_test['status'] = df_test['status'].fillna('Unknown')
df_test['raw_material_format_type'] = df_test['raw_material_format_type'].fillna('Unknown')
df_test['transporter_name'] = df_test.get('transporter_name', pd.Series(index=df_test.index, name='transporter_name')).fillna('Unknown')

# One-hot encode and align with the training columns
print("One-hot encoding and aligning test columns...")
df_test = pd.get_dummies(df_test, columns=['f_is_month_end', 'status', 'raw_material_format_type', 'transporter_name'], prefix_sep='_f_')
X_test_aligned, _ = df_test.align(X_full, join='right', axis=1, fill_value=0)
X_test = X_test_aligned[FEATURE_COLS_FULL] # Ensure exact column order

print("Test feature set created successfully.")
dtest = xgb.DMatrix(X_test)

# --- 6. Predict & Save ---
print("\n--- 6. Making final tuned predictions ---")
final_predictions_tuned = model_final_tuned.predict(dtest)

# Ensure predictions are non-negative
final_predictions_tuned[final_predictions_tuned < 0] = 0

# Create submission dataframe with the CORRECT column name
df_submission_tuned = pd.DataFrame({
    'ID': df_test['ID'],
    'predicted_weight': final_predictions_tuned # <-- CORRECTED COLUMN NAME
})

# Save the file
SUBMISSION_FILE_TUNED = "xgboost.csv"
df_submission_tuned.to_csv(SUBMISSION_FILE_TUNED, index=False)

print(f"\n--- Tuned submission file '{SUBMISSION_FILE_TUNED}' created successfully! ---")

--- Phase 7: Generating Tuned Kaggle Submission (XGBoost v3) ---
Loaded best parameters from 200-trial Optuna study: {'learning_rate': 0.099060649608946, 'max_depth': 6, 'subsample': 0.8172419290767476, 'colsample_bytree': 0.7786050937155478, 'min_child_weight': 7}

Finding best n_estimators for tuned model...
Found best iteration for tuned model: 36

--- 2. Re-training tuned model on all data ---
Training on full dataset with 36 rounds...
Final tuned model trained.

--- 3. Loading all raw data for test set generation ---


/Users/jennarx/Desktop/NTNU/ModernMachineLearning/group_project/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [01:19:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


All raw files loaded.
--- 4. Cleaning raw data ---
Test set grid created. Shape: (30450, 4)
--- 5. Building test set features (v3) ---
Building feature (f_cumulative_po_quantity)...
Building features (Time-Based)...
Building features (Entity & Lag)...
Building features (f_receivals_Nd)...
One-hot encoding and aligning test columns...
Test feature set created successfully.

--- 6. Making final tuned predictions ---

--- Tuned submission file 'xgboost.csv' created successfully! ---
